In [12]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Step 1: Scale your features FIRST
scaler = StandardScaler()
X = scaler.fit_transform(X)  # X is your original feature matrix

# Step 2: Now split the scaled data
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples: {X_test.shape[0]}")

# Step 3: Model fitting
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

NameError: name 'X' is not defined

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr, target_names=['Not Safe','Safe']))

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)
print("Decision Tree Accuracy:", accuracy_score(y_test, y_pred_dt))
print(classification_report(y_test, y_pred_dt, target_names=['Not Safe','Safe']))

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf, target_names=['Not Safe','Safe']))

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(random_state=42)
gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)
print("Gradient Boosting Accuracy:", accuracy_score(y_test, y_pred_gb))
print(classification_report(y_test, y_pred_gb, target_names=['Not Safe','Safe']))

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
y_pred_knn = knn.predict(X_test)
print("KNN Accuracy:", accuracy_score(y_test, y_pred_knn))
print(classification_report(y_test, y_pred_knn, target_names=['Not Safe','Safe']))

In [ ]:
from sklearn.svm import SVC

svm = SVC(probability=True, random_state=42)
svm.fit(X_train, y_train)
y_pred_svm = svm.predict(X_test)
print("SVM Accuracy:", accuracy_score(y_test, y_pred_svm))
print(classification_report(y_test, y_pred_svm, target_names=['Not Safe','Safe']))

In [ ]:
import pandas as pd
import numpy as np

model_names = ['Logistic Regression','Decision Tree','Random Forest','Gradient Boosting','KNN','SVM']
preds       = [y_pred_lr, y_pred_dt, y_pred_rf, y_pred_gb, y_pred_knn, y_pred_svm]
models_list = [lr, dt, rf, gb, knn, svm]

accs = [accuracy_score(y_test, p) for p in preds]
aucs = [roc_auc_score(y_test, m.predict_proba(X_test)[:,1]) for m in models_list]

comparison_df = pd.DataFrame({'Model': model_names, 'Accuracy': accs, 'ROC-AUC': aucs})
comparison_df = comparison_df.sort_values('Accuracy', ascending=False).reset_index(drop=True)
print(comparison_df.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

names = comparison_df['Model'].tolist()
accs_sorted = comparison_df['Accuracy'].tolist()
aucs_sorted = comparison_df['ROC-AUC'].tolist()
x = np.arange(len(names))

fig, ax = plt.subplots(figsize=(10,5))
ax.bar(x - 0.2, accs_sorted, 0.35, label='Accuracy', color='#1D9E75')
ax.bar(x + 0.2, aucs_sorted, 0.35, label='ROC-AUC', color='#534AB7')
ax.set_xticks(x); ax.set_xticklabels(names, rotation=15, ha='right')
ax.set_ylim(0.4, 0.85); ax.set_ylabel('Score')
ax.set_title('Model Comparison: Accuracy vs ROC-AUC')
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
import seaborn as sns

cm = confusion_matrix(y_test, y_pred_svm)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not Safe','Safe'], yticklabels=['Not Safe','Safe'])
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.title('Confusion Matrix — SVM')
plt.tight_layout(); plt.show()

In [ ]:
from sklearn.metrics import roc_curve

fig, ax = plt.subplots(figsize=(8,6))
colors = ['#1D9E75','#534AB7','#D85A30','#BA7517','#3266ad','#993556']
for model, name, col in zip(models_list, model_names, colors):
    fpr, tpr, _ = roc_curve(y_test, model.predict_proba(X_test)[:,1])
    auc = roc_auc_score(y_test, model.predict_proba(X_test)[:,1])
    ax.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})", color=col)
ax.plot([0,1],[0,1],'k--', alpha=0.4)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — All Models'); ax.legend(loc='lower right')
plt.tight_layout(); plt.show()

In [ ]:
feature_names = df.drop(['Potability','ph_category'], axis=1).columns
importances = pd.Series(rf.feature_importances_, index=feature_names).sort_values(ascending=True)

plt.figure(figsize=(7,5))
importances.plot.barh(color='#1D9E75')
plt.title('Feature Importance — Random Forest')
plt.xlabel('Importance Score')
plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(9,7))
corr = df[feature_names.tolist()+['Potability']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap')
plt.tight_layout(); plt.show()

In [ ]:
from sklearn.model_selection import cross_val_score

for model, name in zip(models_list, model_names):
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    print(f"{name}: CV Mean={cv_scores.mean():.4f}, Std={cv_scores.std():.4f}")